# HDO Catalyst ML and Optimization

Demonstrates catalyst screening and hydrodeoxygenation optimization for oxygen removal, water formation, HHV and DOD-related outputs.


In [ ]:
import pandas as pd

data = [
    # catalyst, metal_wt, SA, PV, PD, SiAl, T, cat_load, time, P,
    # O_wt, H2O_wt, HHV, visc, TAN, DOD
    ["Fe2O3_HZ", 1.27, 12.9, 0.070, 3.13, 6.94, 250, 2.5, 2.0, 1,
     53.8, 27.0, 17.7, 2.31, 139, 73.4],

    ["ZnO_HZ", 1.30, 50.3, 0.077, 3.33, 6.63, 250, 2.5, 2.0, 1,
     40.5, 10.1, 20.9, 2.52, 166, 85.2],

    ["CuO_HZ", 2.32, 30.7, 0.086, 3.10, 5.69, 250, 2.5, 2.0, 1,
     40.2, 8.3, 20.9, 3.13, 118, 85.4],

    # Temperature sweep (ZnO/H-Z)
    ["ZnO_HZ", 1.30, 50.3, 0.077, 3.33, 6.63, 275, 2.5, 2.0, 1,
     36.2, 8.9, 22.1, 2.35, 150, 88.1],

    ["ZnO_HZ", 1.30, 50.3, 0.077, 3.33, 6.63, 300, 2.5, 2.0, 1,
     32.8, 7.5, 21.0, 2.10, 160, 90.2],

    # Catalyst loading sweep (ZnO/H-Z)
    ["ZnO_HZ", 1.30, 50.3, 0.077, 3.33, 6.63, 275, 5.0, 2.0, 1,
     34.5, 7.9, 22.6, 2.20, 142, 89.5],

    ["ZnO_HZ", 1.30, 50.3, 0.077, 3.33, 6.63, 275, 7.5, 2.0, 1,
     31.0, 6.8, 21.8, 2.05, 155, 91.0],
]

cols = [
    "Catalyst","Metal_wt","SurfaceArea","PoreVolume","PoreDiameter",
    "SiAl","Temp_C","CatLoading_wt","Time_h","Pressure_atm",
    "Oxygen_wt","Water_wt","HHV_MJkg","Viscosity_cP","TAN","DOD_pct"
]

df = pd.DataFrame(data, columns=cols)
df.to_csv("HDO_real_dataset.csv", index=False)
df


R-100 (acid) → R-200 (acid) → R-300 (HDO catalyst)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.preprocessing import StandardScaler

# Encode catalyst as integer
cat_map = {c:i for i,c in enumerate(df["Catalyst"].unique())}
df["CatID"] = df["Catalyst"].map(cat_map)

X = df[[
    "CatID","Metal_wt","SurfaceArea","PoreVolume","PoreDiameter",
    "SiAl","Temp_C","CatLoading_wt"
]].values

Y = df[[
    "Oxygen_wt","Water_wt","HHV_MJkg","Viscosity_cP","DOD_pct"
]].values

sx, sy = StandardScaler(), StandardScaler()
Xn, Yn = sx.fit_transform(X), sy.fit_transform(Y)

Xn = torch.tensor(Xn, dtype=torch.float32)
Yn = torch.tensor(Yn, dtype=torch.float32)

class HDO_GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(Xn.shape[1], 64)
        self.fc2 = nn.Linear(64, 64)
        self.out = nn.Linear(64, Yn.shape[1])

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

model = HDO_GNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(2000):
    pred = model(Xn)
    loss = F.mse_loss(pred, Yn)
    opt.zero_grad(); loss.backward(); opt.step()

print("Training complete")


In [ ]:
def predict_kpi(catalyst, T, loading):
    row = df[df["Catalyst"]==catalyst].iloc[0]
    x = np.array([[
        cat_map[catalyst],
        row.Metal_wt, row.SurfaceArea, row.PoreVolume,
        row.PoreDiameter, row.SiAl, T, loading
    ]])
    x = torch.tensor(sx.transform(x), dtype=torch.float32)
    y = sy.inverse_transform(model(x).detach().numpy())[0]
    return {
        "Oxygen_wt": round(y[0],2),
        "Water_wt": round(y[1],2),
        "HHV": round(y[2],2),
        "Viscosity": round(y[3],2),
        "DOD": round(y[4],2)
    }

# DEMO
print("Base:", predict_kpi("ZnO_HZ", 250, 2.5))
print("Temp ↑:", predict_kpi("ZnO_HZ", 300, 2.5))
print("Catalyst loading ↑:", predict_kpi("ZnO_HZ", 275, 5.0))
print("Change catalyst:", predict_kpi("CuO_HZ", 275, 2.5))


In [ ]:
!pip -q install pymoo
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import numpy as np

class HDO_Opt(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=3, n_obj=4,
                         xl=[250,2.5,0],
                         xu=[300,7.5,len(cat_map)-1])

    def _evaluate(self, x, out, *args, **kwargs):
        T, load, catid = x
        cat = list(cat_map.keys())[int(round(catid))]
        kpi = predict_kpi(cat, T, load)
        out["F"] = [
            kpi["Oxygen_wt"],
            kpi["Water_wt"],
            kpi["Viscosity"],
            -kpi["DOD"]
        ]

algo = NSGA2(pop_size=80)
res = minimize(HDO_Opt(), algo, ("n_gen", 100), verbose=False)

res.X[:5]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 140


In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = "HDO_ML_Dataset.csv"   # if you uploaded it
df = pd.read_csv(CSV_PATH)

print(df.columns)
display(df)


In [ ]:
df2 = df[df["Catalyst"]!="Raw"].copy()
x = np.arange(len(df2))

plt.figure(figsize=(8,4))
plt.bar(x-0.2, df2["Oxygen_wt"], width=0.2, label="Oxygen_wt")
plt.bar(x,     df2["Water_wt"],  width=0.2, label="Water_wt")
plt.bar(x+0.2, df2["DOD_pct"],   width=0.2, label="DOD_%")
plt.xticks(x, df2["Catalyst"], rotation=25, ha="right")
plt.title("HDO KPIs (from extracted Table 4)")
plt.grid(axis="y")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
!pip -q install torch

import torch
import torch.nn as nn
import torch.nn.functional as F

train_df = df[df["Catalyst"]!="Raw"].copy().reset_index(drop=True)

cats = train_df["Catalyst"].unique().tolist()
cat_to_id = {c:i for i,c in enumerate(cats)}
train_df["CatID"] = train_df["Catalyst"].map(cat_to_id)

# Inputs at R-300 (catalyst + setpoints)
X_num = train_df[["Si/Al","SurfaceArea_m2g","PoreVolume_ccg","PoreDiameter_nm","Temp_C","CatLoading_wt"]].values.astype(np.float32)
cat_ids = train_df["CatID"].values.astype(np.int64)

# Targets (KPIs)
Y_cols = ["Oxygen_wt","Water_wt","HHV_MJkg","Viscosity_cP","DOD_pct"]
Y = train_df[Y_cols].values.astype(np.float32)

# Normalize
X_mu, X_sig = X_num.mean(axis=0), X_num.std(axis=0) + 1e-6
Y_mu, Y_sig = Y.mean(axis=0), Y.std(axis=0) + 1e-6
Xn = (X_num - X_mu)/X_sig
Yn = (Y - Y_mu)/Y_sig

# Chain adjacency (4 nodes)
N=4
A = np.zeros((N,N), dtype=np.float32)
for u,v in [(0,1),(1,2),(2,3)]:
    A[u,v]=1; A[v,u]=1
A += np.eye(N, dtype=np.float32)
A = A / (A.sum(axis=1, keepdims=True)+1e-6)
A = torch.tensor(A, dtype=torch.float32)

def build_nodes(x_row):
    nodes = np.zeros((N,6), dtype=np.float32)
    nodes[2,:] = x_row  # only R-300 has features
    return nodes

X_nodes = torch.tensor(np.stack([build_nodes(Xn[i]) for i in range(len(Xn))]), dtype=torch.float32)
cat_ids_t = torch.tensor(cat_ids, dtype=torch.long)
Yn_t = torch.tensor(Yn, dtype=torch.float32)

class ChainGNN(nn.Module):
    def __init__(self, n_cat, emb=8, hid=64, out_dim=5):
        super().__init__()
        self.emb = nn.Embedding(n_cat, emb)
        self.lin_in = nn.Linear(6+emb, hid)
        self.lin_mp = nn.Linear(hid, hid)
        self.lin_out = nn.Linear(hid, out_dim)

    def forward(self, x_nodes, cat_id, A):
        B,N,_ = x_nodes.shape
        e = self.emb(cat_id)[:,None,:].repeat(1,N,1)
        h = torch.cat([x_nodes, e], dim=-1)
        h = F.relu(self.lin_in(h))
        h = torch.matmul(A, h)
        h = F.relu(self.lin_mp(h))
        h = torch.matmul(A, h)
        return self.lin_out(h[:,3,:])

model = ChainGNN(n_cat=len(cats), emb=8, hid=64, out_dim=len(Y_cols))
opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-4)

for epoch in range(1, 1201):
    pred = model(X_nodes, cat_ids_t, A)
    loss = F.mse_loss(pred, Yn_t)
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 300 == 0:
        print("epoch", epoch, "loss", float(loss))

print("trained")


In [ ]:
!pip -q install ipywidgets

import ipywidgets as widgets
from IPython.display import display

def predict_kpi(catalyst, T=250.0, load=2.5):
    base = train_df[train_df["Catalyst"]==catalyst].iloc[0]
    x_num = np.array([
        base["Si/Al"], base["SurfaceArea_m2g"], base["PoreVolume_ccg"], base["PoreDiameter_nm"],
        T, load
    ], dtype=np.float32)

    x_n = (x_num - X_mu)/X_sig
    nodes = np.zeros((1,4,6), dtype=np.float32)
    nodes[0,2,:] = x_n
    nodes_t = torch.tensor(nodes, dtype=torch.float32)
    cid = torch.tensor([cat_to_id[catalyst]], dtype=torch.long)

    with torch.no_grad():
        y_n = model(nodes_t, cid, A).numpy()[0]
    y = y_n * Y_sig + Y_mu
    return dict(zip(Y_cols, y.tolist()))

def live(catalyst, T, load):
    k = predict_kpi(catalyst, float(T), float(load))
    print({kk: round(v,3) for kk,v in k.items()})

    Ts = np.linspace(230, 320, 19)
    oxy = [predict_kpi(catalyst, float(t), float(load))["Oxygen_wt"] for t in Ts]
    dod = [predict_kpi(catalyst, float(t), float(load))["DOD_pct"] for t in Ts]

    plt.figure(figsize=(7,4))
    plt.plot(Ts, oxy, label="Oxygen_wt")
    plt.plot(Ts, dod, label="DOD_%")
    plt.title(f"{catalyst} | load={load} wt%")
    plt.xlabel("Temperature (°C)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

w_cat = widgets.Dropdown(options=cats, value=cats[0], description="Catalyst")
w_T   = widgets.FloatSlider(value=250, min=230, max=320, step=1, description="T (°C)")
w_L   = widgets.FloatSlider(value=2.5, min=2.5, max=10.0, step=0.5, description="Load")

ui = widgets.VBox([w_cat, w_T, w_L])
out = widgets.interactive_output(live, {"catalyst": w_cat, "T": w_T, "load": w_L})
display(ui, out)


In [ ]:
!pip -q install pymoo

import numpy as np
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

class HDO_MOO(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=3, n_obj=4, n_constr=0,
                         xl=np.array([230.0, 2.5, 0.0]),
                         xu=np.array([320.0, 10.0, float(len(cats)-1)]))

    def _evaluate(self, x, out, *args, **kwargs):
        T, load, cid = x
        cid = int(np.clip(np.round(cid), 0, len(cats)-1))
        cat = cats[cid]
        k = predict_kpi(cat, float(T), float(load))
        out["F"] = np.array([
            k["Oxygen_wt"],
            k["Water_wt"],
            k["Viscosity_cP"],
            -k["DOD_pct"]
        ], dtype=float)

res = minimize(HDO_MOO(), NSGA2(pop_size=80), ("n_gen", 120), seed=1, verbose=False)

print("First 10 Pareto samples:")
for i in range(min(10, len(res.X))):
    T, load, cid = res.X[i]
    cid = int(np.round(cid))
    cat = cats[cid]
    print(i, cat, "T=",round(T,1), "load=",round(load,2), {k: round(v,2) for k,v in predict_kpi(cat,T,load).items()})


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

F = res.F  # shape: (n_solutions, 4)
# F columns: [Oxygen_wt, Water_wt, Viscosity, -DOD]
oxygen = F[:,0]
water  = F[:,1]
visc   = F[:,2]
neg_dod= F[:,3]
dod    = -neg_dod

# Optional: pick a "best compromise" point by a simple normalized score
Fn = (F - F.min(axis=0)) / (F.ptp(axis=0) + 1e-9)
score = Fn[:,0] + Fn[:,1] + 0.2*Fn[:,2] + 0.2*Fn[:,3]   # tune weights
best_i = int(np.argmin(score))

def scatter2(x, y, xlabel, ylabel, title, best_idx=None):
    plt.figure(figsize=(6.5,4.5))
    plt.scatter(x, y, s=18)
    if best_idx is not None:
        plt.scatter([x[best_idx]], [y[best_idx]], s=120, marker="*", label="Best-compromise")
        plt.legend()
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

scatter2(oxygen, dod,  "Oxygen_wt (min)", "DOD_% (max)", "Pareto: Oxygen vs DOD", best_i)
scatter2(water,  dod,  "Water_wt (min)",  "DOD_% (max)", "Pareto: Water vs DOD", best_i)
scatter2(oxygen, visc, "Oxygen_wt (min)", "Viscosity (min)", "Pareto: Oxygen vs Viscosity", best_i)
scatter2(water,  visc, "Water_wt (min)",  "Viscosity (min)", "Pareto: Water vs Viscosity", best_i)

print("Best-compromise decision variables:", res.X[best_i])
print("Best-compromise objectives [O2, H2O, Visc, -DOD]:", res.F[best_i])


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 140


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

F = res.F  # shape: (n_solutions, 4)
# F columns: [Oxygen_wt, Water_wt, Viscosity, -DOD]
oxygen = F[:,0]
water  = F[:,1]
visc   = F[:,2]
neg_dod= F[:,3]
dod    = -neg_dod

# Optional: pick a "best compromise" point by a simple normalized score
Fn = (F - F.min(axis=0)) / (np.ptp(F, axis=0) + 1e-9)
score = Fn[:,0] + Fn[:,1] + 0.2*Fn[:,2] + 0.2*Fn[:,3]   # tune weights
best_i = int(np.argmin(score))

def scatter2(x, y, xlabel, ylabel, title, best_idx=None):
    plt.figure(figsize=(6.5,4.5))
    plt.scatter(x, y, s=18)
    if best_idx is not None:
        plt.scatter([x[best_idx]], [y[best_idx]], s=120, marker="*", label="Best-compromise")
        plt.legend()
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

scatter2(oxygen, dod,  "Oxygen_wt (min)", "DOD_% (max)", "Pareto: Oxygen vs DOD", best_i)
scatter2(water,  dod,  "Water_wt (min)",  "DOD_% (max)", "Pareto: Water vs DOD", best_i)
scatter2(oxygen, visc, "Oxygen_wt (min)", "Viscosity (min)", "Pareto: Oxygen vs Viscosity", best_i)
scatter2(water,  visc, "Water_wt (min)",  "Viscosity (min)", "Pareto: Water vs Viscosity", best_i)

print("Best-compromise decision variables:", res.X[best_i])
print("Best-compromise objectives [O2, H2O, Visc, -DOD]:", res.F[best_i])

In [ ]:
# Show top 10 by the same compromise score
top = np.argsort(score)[:10]
for j,i in enumerate(top):
    T, load, cid = res.X[i]
    cid = int(np.round(cid))
    cat = cats[cid]
    print(j, "|", cat, "| T=", round(T,1), "load=", round(load,2),
          "| O2=", round(oxygen[i],2),
          "H2O=", round(water[i],2),
          "Visc=", round(visc[i],2),
          "DOD=", round(dod[i],2))


In [ ]:
import pandas as pd

data = [
    ("Fe2O3/H-Z", 279.3, 6.67, 53.8, 27.0, 17.7, 2.31, 73.4),
    ("CuO/H-Z",   278.3, 8.21, 40.19, 8.29, 20.9, 3.13, 85.41),
    ("CuO/H-Z",   282.4, 9.46, 40.19, 8.29, 20.9, 3.13, 85.41),
    ("Fe2O3/H-Z", 306.5, 4.96, 53.8, 27.0, 17.7, 2.31, 73.4),
    ("ZnO/H-Z",   253.2, 5.8,  40.51,10.11,20.9, 2.52, 85.19),
    ("ZnO/H-Z",   250.3, 2.67, 40.51,10.11,20.9, 2.52, 85.19),
    ("ZnO/H-Z",   278.7, 9.4,  40.51,10.11,20.9, 2.52, 85.19),
    ("Fe2O3/H-Z", 312.6, 5.46, 53.8, 27.0, 17.7, 2.31, 73.4),
    ("Fe2O3/H-Z", 253.4, 4.57, 53.8, 27.0, 17.7, 2.31, 73.4),
    ("Fe2O3/H-Z", 244.1, 4.96, 53.8, 27.0, 17.7, 2.31, 73.4),
]
dfp = pd.DataFrame(data, columns=["Catalyst","T","Load","Oxygen","Water","HHV","Viscosity","DOD"])
dfp


In [ ]:
import numpy as np

labels = [f"{c}\n{int(T)}°C\n{l:.1f} wt%" for c,T,l in zip(dfp.Catalyst, dfp.T, dfp.Load)]

# Normalize each KPI to 0–1 (robust min-max)
def minmax(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

O2n  = minmax(dfp["Oxygen"])
H2On = minmax(dfp["Water"])
Vn   = minmax(dfp["Viscosity"])
DODn = minmax(dfp["DOD"])
HHVn = minmax(dfp["HHV"])

# Score: smaller is better
# (Tune weights if needed)
score = 0.35*O2n + 0.25*H2On + 0.15*Vn - 0.15*DODn - 0.10*HHVn
best_i = int(np.argmin(score))

best_i, dfp.loc[best_i, ["Catalyst","T","Load","Oxygen","Water","HHV","Viscosity","DOD"]]


In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(dfp))

plt.figure(figsize=(12,5))

# Green shades (benefits)
plt.bar(x, dfp["DOD"], color="#2ca25f", label="DOD (%)")
plt.bar(x, dfp["HHV"], bottom=dfp["DOD"], color="#66c2a4", label="HHV (MJ/kg)")

# Blue shades (penalties)
bottom_pen = dfp["DOD"] + dfp["HHV"]
plt.bar(x, dfp["Oxygen"], bottom=bottom_pen, color="#3182bd", label="Oxygen wt%")
plt.bar(x, dfp["Water"], bottom=bottom_pen + dfp["Oxygen"], color="#6baed6", label="Water wt%")
plt.bar(x, dfp["Viscosity"], bottom=bottom_pen + dfp["Oxygen"] + dfp["Water"],
        color="#9ecae1", label="Viscosity (cP)")

# Highlight best
plt.scatter([best_i], [bottom_pen.iloc[best_i] + dfp["Oxygen"].iloc[best_i] + dfp["Water"].iloc[best_i] + dfp["Viscosity"].iloc[best_i]],
            s=180, marker="*", color="black", label="Best-compromise")

plt.xticks(x, labels)
plt.ylabel("Stacked KPI (raw units mixed)")
plt.title("Pareto Points – RAW Stacked KPIs (green=benefits, blue=penalties)")
plt.legend(ncol=3, frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
!apt-get -qq install graphviz
!pip -q install graphviz

from graphviz import Digraph

dot = Digraph("PID_Flow", format="png")
dot.attr(rankdir="TB", bgcolor="white")

# Nodes
dot.node("S", "START", shape="oval", style="filled", fillcolor="black", fontcolor="white")
dot.node("A", "P&ID Source (DWG)\n• Original engineering drawing\n• Human readable", shape="box", style="rounded,filled", fillcolor="#e3f2fd")
dot.node("B", "Convert DWG → DXF\n• Export CAD geometry\n• Entities: LINE/POLYLINE/INSERT/TEXT", shape="box", style="rounded,filled", fillcolor="#e3f2fd")
dot.node("C", "Parse DXF (ezdxf)\nExtract:\n• Pipes (LINE/POLYLINE)\n• Equip symbols (INSERT)\n• Tags (TEXT)\n• Layers = process/signal", shape="box", style="rounded,filled", fillcolor="#fff3e0")
dot.node("D", "Build P&ID JSON\nnodes[] + edges[]\n• node: id,type,(x,y)\n• edge: from,to,type", shape="box", style="rounded,filled", fillcolor="#fff3e0")
dot.node("E", "Normalize + Clean Geometry\n• remove outliers\n• rescale coords", shape="box", style="rounded,filled", fillcolor="#f3e5f5")
dot.node("F", "KNN Connectivity (sanity)\n• local edges only\n• remove long-range false links", shape="box", style="rounded,filled", fillcolor="#f3e5f5")
dot.node("G", "ML-ready Graph\nG=(V,E)\n• nodes: equipment/reactors/instruments\n• edges: process pipes + signal lines", shape="box", style="rounded,filled", fillcolor="#e0f7fa")
dot.node("H", "GNN + pyMOO\n• learn KPI surrogate\n• Pareto-optimal setpoints", shape="box", style="rounded,filled", fillcolor="#e0f7fa")
dot.node("T", "STOP", shape="oval", style="filled", fillcolor="black", fontcolor="white")

# Edges
dot.edge("S","A")
dot.edge("A","B")
dot.edge("B","C")
dot.edge("C","D")
dot.edge("D","E")
dot.edge("E","F")
dot.edge("F","G")
dot.edge("G","H")
dot.edge("H","T")

dot

In [ ]:
from google.colab import files

# Render the dot object to a PNG file
dot.render('PID_Flow_Workflow', format='png', view=False)

# Download the generated PNG file
files.download('PID_Flow_Workflow.png')